# VoyageAssistant — Serveur Conversationnel Multilingue de Réservation de Vols

**Projet** : Développement et déploiement d'un serveur conversationnel  
**Application** : Système de recherche et réservation de vols  
**Langues** : Français · English · 中文  
**Données** : `flights_database_v2.json` (300 vols, 60 villes, 15 compagnies)  
**Modèle LLM** : Llama-3.2-3B-Instruct (local ou HuggingFace)

---

## Modèle de tâche HTA (Hierarchical Task Analysis)

```
0. Réserver un vol
├── 1. Accueillir l'utilisateur
├── 2. Collecter les critères
│   ├── 2.1 Identifier ville de départ (from_city) [OBLIGATOIRE]
│   ├── 2.2 Identifier ville d'arrivée (to_city) [OBLIGATOIRE]
│   ├── 2.3 Identifier la date de départ
│   ├── 2.4 Identifier la classe (économique / affaires / première)
│   ├── 2.5 Identifier le type de trajet (aller simple / aller-retour)
│   ├── 2.6 Identifier la compagnie aérienne
│   ├── 2.7 Identifier le nombre de passagers
│   └── 2.8 Identifier le budget / critère de prix
├── 3. Rechercher les vols disponibles
│   ├── 3.1 Filtrer par critères obligatoires
│   ├── 3.2 Filtrer par critères optionnels
│   └── 3.3 Trier et présenter les résultats
├── 4. Présenter les résultats
│   ├── 4.1 Afficher le vol recommandé
│   ├── 4.2 Proposer des alternatives (BF=7)
│   └── 4.3 Fournir les détails complets
└── 5. Confirmer la réservation
    ├── 5.1 Demander confirmation
    ├── 5.2 Gérer acceptation / rejet
    └── 5.3 Enregistrer la réservation
```


## 1. Installation des dépendances

In [98]:
# Installation des dépendances
!pip install -q transformers accelerate bitsandbytes huggingface_hub

from huggingface_hub import login
login(token="  ")

## 2. Imports et configuration

In [99]:
import json
import re
import os
import random
import unicodedata
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login
import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")


Device utilisé : cuda
GPU disponible : True
GPU : Tesla T4


## 3. Chargement du modèle Llama-3.2-3B-Instruct

Le modèle est chargé en mode `float16` pour optimiser la mémoire GPU.  
Il est cherché d'abord en local (paths Colab standards), puis téléchargé depuis HuggingFace si nécessaire.


In [100]:
# Initialisation du modèle (identique à 1.3)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

def init_LLM():
    """Initialise Llama-3.2-3B-Instruct"""
    MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

    print("Chargement du tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device utilisé : {device}")

    print("Chargement du modèle...")
    if device == "cuda":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            device_map={"": "cpu"},
            trust_remote_code=True
        )

    print("✅ Modèle chargé")
    return tokenizer, model, device

tokenizer, model, device = init_LLM()

# ── Création du pipeline (utilisé par NLU et NLG) ──────────────────
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
)
print("✅ llm_pipeline prêt")


Chargement du tokenizer...
Device utilisé : cuda
Chargement du modèle...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

✅ Modèle chargé
✅ llm_pipeline prêt


In [101]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Chargement des données

Chargement de la base de vols (300 vols, 60 villes, 15 compagnies) et du lexique actif multilingue.


In [102]:
def upload_or_load_json(filename, description="fichier"):
    """Charge un JSON depuis le répertoire courant, Google Drive, ou upload Colab."""
    paths_to_try = [
        filename,
        f"/content/{filename}",
        f"/content/drive/MyDrive/{filename}",
        f"/content/drive/MyDrive/conversation/{filename}",
        f"/content/drive/MyDrive/Projet_2026/{filename}",
    ]
    for path in paths_to_try:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            print(f"✓ {description} chargé depuis : {path}")
            return data

    try:
        from google.colab import files
        print(f"⬆ Veuillez uploader le fichier {filename}")
        uploaded = files.upload()
        if filename in uploaded:
            data = json.loads(uploaded[filename].decode("utf-8"))
            print(f"✓ {description} chargé depuis l'upload")
            return data
    except ImportError:
        pass

    raise FileNotFoundError(f"Impossible de trouver {filename}")

flights_data = upload_or_load_json("flights_database_v2.json", "Base de vols (300 vols)")
lexique     = upload_or_load_json("lexique_voyage_v2.json",   "Lexique actif multilingue (FR/EN/ZH)")

flights_db = flights_data["flights"]
cities_db  = flights_data["cities"]
airlines_db = flights_data["airlines"]

print(f"\n📊 Statistiques :")
print(f"  - Vols en base         : {len(flights_db)}")
print(f"  - Villes disponibles   : {len(cities_db)}")
print(f"  - Compagnies           : {len(airlines_db)}")
print(f"  - Intents reconnus     : {len(lexique['intents'])}")
print(f"  - Slots disponibles    : {len(lexique['slots_reconnus'])}")


✓ Base de vols (300 vols) chargé depuis : /content/drive/MyDrive/conversation/flights_database_v2.json
✓ Lexique actif multilingue (FR/EN/ZH) chargé depuis : /content/drive/MyDrive/conversation/lexique_voyage_v2.json

📊 Statistiques :
  - Vols en base         : 1224
  - Villes disponibles   : 60
  - Compagnies           : 15
  - Intents reconnus     : 11
  - Slots disponibles    : 24


## 5. Module NLU — Compréhension du langage naturel (multilingue)

Le module NLU (Natural Language Understanding) est responsable de :
1. **Détecter la langue** du message (FR / EN / ZH)
2. **Analyser par règles** (grammaires regex) pour les cas simples et fréquents
3. **Appeler le LLM** (Llama-3.2-3B-Instruct) pour les cas ambigus

### Grammaires de reconnaissance (étape 4)

Les grammaires regex ci-dessous modélisent des fragments de langue complexes :
- `DATE_GRAMMAR` : dates absolues et relatives (JJ/MM/AAAA, "le 15 mars", "next Monday"...)
- `TIME_GRAMMAR` : heures en FR/EN/ZH (13h45, 13:45, 1:45pm, 13点45分...)
- `CONFIRMATION_GRAMMAR` : toutes formes d'acquiescement en 3 langues
- `NEGATION_GRAMMAR` : toutes formes de refus / négation en 3 langues
- `PRICE_GRAMMAR` : expressions de budget (moins de X$, entre X et Y, under $300...)
- `PASSENGER_GRAMMAR` : nombre de passagers (2 personnes, for 3 travelers, 2名乘客...)
- `CITY_GRAMMAR` : capture de couples ville-départ/arrivée
- `CORRECTION_GRAMMAR` : détection de corrections de slot


In [126]:
INTENT_LIST = [
    "salutation", "flight", "airfare", "flight_time", "aircraft",
    "airline", "ground_service", "accepter", "rejeter",
    "correction_slot", "demande_clarification", "out_of_scope", "au_revoir"
]

SLOT_NAMES = [
    "from_city", "to_city", "depart_day", "depart_month",
    "depart_date_day_number", "depart_time", "depart_period",
    "depart_date_relative", "airline", "class_type", "round_trip",
    "flight_stop", "cost_relative", "fare_amount", "flight_mod",
    "arrive_time", "stop_city", "passenger_count",
    "return_day", "return_date_day_number", "return_month", "return_time"
]

# ─── Détection de langue ────────────────────────────────────────────

def strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def detect_language(message: str) -> str:
    """Détecte la langue : 'fr', 'en', ou 'zh'."""
    for ch in message:
        if '\u4e00' <= ch <= '\u9fff' or '\u3400' <= ch <= '\u4dbf':
            return "zh"
    zh_markers = ["我", "你", "航班", "想", "飞", "好的", "谢谢", "再见", "预订", "出发"]
    if any(m in message for m in zh_markers):
        return "zh"
    msg_low = message.lower()
    en_patterns = [
        r"\bhello\b", r"\bhi\b", r"\bgoodbye\b", r"\bbye\b", r"\bthanks\b",
        r"\bflight\b", r"\bticket\b", r"\bbook\b", r"\bwant\b", r"\bneed\b",
        r"\bfrom\b", r"\bto\b", r"\bplane\b", r"\bairline\b", r"\bcheap\b",
    ]
    if any(re.search(p, msg_low) for p in en_patterns):
        return "en"
    en_markers = [r"\bi\b", r"\byou\b", r"\bthe\b", r"\band\b", r"\bwith\b"]
    if sum(1 for p in en_markers if re.search(p, msg_low)) >= 2:
        return "en"
    return "fr"

# ─── Grammaires de reconnaissance complexes ─────────────────────────

# Date absolue : 15/03, 15-03-2025, March 15, 15 mars 2025, 3月15日
DATE_GRAMMAR = re.compile(
    r"\b(?P<day>\d{1,2})\s*[/\-.\s]\s*(?P<month_num>\d{1,2})(?:\s*[/\-.]\s*(?P<year>\d{2,4}))?\b"
    r"|(?P<day2>\d{1,2})\s*(?P<month_fr>janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)(?:\s+(?P<year2>\d{4}))?"
    r"|(?P<month_en>january|february|march|april|may|june|july|august|september|october|november|december)\s+(?P<day3>\d{1,2})(?:st|nd|rd|th)?(?:,?\s*(?P<year3>\d{4}))?"
    r"|(?P<cn_month>\d{1,2})月(?P<cn_day>\d{1,2})(?:日|号)",
    re.IGNORECASE
)

# Heure : 13h45, 13:45, 1:45pm, 13点45分, 下午1点
TIME_GRAMMAR = re.compile(
    r"(?P<h1>\d{1,2})h(?P<m1>\d{0,2})"
    r"|(?P<h2>\d{1,2}):(?P<m2>\d{2})\s*(?P<ampm>am|pm)?"
    r"|(?P<h3>\d{1,2})\s*heures?\s*(?P<m3>\d{0,2})"
    r"|(?P<h4>\d{1,2})\s*o'clock"
    r"|(?P<prefix>上午|下午)?(?P<h5>\d{1,2})\s*[点時时]\s*(?P<m5>\d{0,2})(?:分)?",
    re.IGNORECASE
)

# Date relative : demain, next week, 下周, dans 3 jours...
DATE_RELATIVE_GRAMMAR = re.compile(
    r"\b(aujourd'hui|demain|après-demain|ce\s+week-end|la\s+semaine\s+prochaine|le\s+mois\s+prochain"
    r"|dans\s+\d+\s+(?:jours?|semaines?|mois)"
    r"|lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche"
    r"|today|tomorrow|next\s+(?:week|month|monday|tuesday|wednesday|thursday|friday|saturday|sunday)"
    r"|今天|明天|后天|下周|下个月|本周末"
    r"|周[一二三四五六日]|星期[一二三四五六日])\b",
    re.IGNORECASE
)

# Confirmation : oui, ok, d'accord, yes, 好的...
CONFIRMATION_GRAMMAR = re.compile(
    r"^\s*(?:oui|ok|okay|d'accord|parfait|bien\s+sûr|absolument|tout\s+à\s+fait"
    r"|c'est\s+ça|exactement|volontiers|avec\s+plaisir|super|génial|allez|je\s+confirme"
    r"|yes|yeah|yep|sure|of\s+course|absolutely|right|correct|perfect|great|alright|confirmed|let's\s+do\s+it"
    r"|好的|是的|对|没问题|当然|好|可以|行|没错|确认|同意|好吧|嗯|确定|就这个)\s*[!.]*\s*$",
    re.IGNORECASE
)

# Négation : non, no, 不...
NEGATION_GRAMMAR = re.compile(
    r"^\s*(?:non|pas\s+du\s+tout|non\s+merci|je\s+ne\s+pense\s+pas|plutôt\s+pas|certainement\s+pas|jamais|pas\s+vraiment"
    r"|no|nope|not\s+really|no\s+thanks|never|not\s+at\s+all|don't\s+want|not\s+interested|absolutely\s+not|negative"
    r"|不|不是|不要|不行|算了|不用了|不想|拒绝|不合适|不对|没有)\s*[!.]*\s*$",
    re.IGNORECASE
)

# Prix : moins de 200$, under 300 dollars, 300以下, entre 150 et 250€
PRICE_GRAMMAR = re.compile(
    r"(?:moins\s+de|pas\s+plus\s+de|under|below|不超过|少于|\<)\s*(\d+)\s*(?:dollars?|\$|euros?|€|元|美元)?"
    r"|(?:plus\s+de|more\s+than|超过|多于|\>)\s*(\d+)\s*(?:dollars?|\$|euros?|€|元|美元)?"
    r"|(?:entre|between|在)\s*(\d+)\s*(?:et|and|到|-)\s*(\d+)\s*(?:dollars?|\$|euros?|€|元|美元)?"
    r"|(?:environ|autour\s+de|around|about|大约|左右)\s*(\d+)\s*(?:dollars?|\$|euros?|€|元|美元)?"
    r"|(\d+)\s*(?:dollars?|\$|euros?|€|元|美元)",
    re.IGNORECASE
)

# Passagers : 2 personnes, for 3 travelers, 2名乘客, 3张票
PASSENGER_GRAMMAR = re.compile(
    r"(\d+)\s*(?:personnes?|passagers?|adultes?|voyageurs?|billets?|places?)"
    r"|(\d+)\s*(?:persons?|passengers?|adults?|travelers?|people|tickets?)"
    r"|(\d+)\s*(?:人|位|名|张票?|个人)",
    re.IGNORECASE
)

# Correction de slot : "non pas X, c'est Y", "je veux dire Y", "en fait Y"
CORRECTION_GRAMMAR = re.compile(
    r"(?:non\s+pas|pas|instead\s+of|not)\s+\w+"
    r"|(?:je\s+(?:voulais?|veux)\s+dire|en\s+fait|actually|je\s+me\s+suis\s+trompé|rectification|correction|update)"
    r"|(?:其实|我是说|不对|纠正)",
    re.IGNORECASE
)

# Ville de départ/arrivée : "de Paris à Tokyo", "from NYC to LA"
CITY_PAIR_GRAMMAR = re.compile(
    r"(?:de|depuis|from|从|由)\s+([A-Za-zÀ-ÿ\s]+?)\s+(?:à|vers|pour|to|去|飞往|到)\s+([A-Za-zÀ-ÿ\s]+?)(?:\s|$|,|\.|en\s|le\s|pour\s)",
    re.IGNORECASE
)

# Période de la journée : matin, après-midi, soir, morning, 早上...
PERIOD_GRAMMAR = re.compile(
    r"\b(tôt\s+le\s+matin|matin|après-midi|fin\s+d'après-midi|soir|nuit|midi"
    r"|early\s+morning|morning|afternoon|late\s+afternoon|evening|night|noon|early"
    r"|早上|上午|清晨|早间|下午|下午晚些时候|晚上|夜间|中午)\b",
    re.IGNORECASE
)

# ─── Résolution de ville ─────────────────────────────────────────────

def resolve_city(text: str, lexique: dict) -> Optional[str]:
    """Tente de résoudre un nom de ville en nom canonique via le lexique."""
    text_clean = text.strip().lower()
    villes = lexique.get("villes", {})
    # Correspondance directe
    for city_name, info in villes.items():
        if city_name.lower() == text_clean:
            return city_name
        # IATA
        if info.get("iata", "").lower() == text_clean:
            return city_name
        # Aliases
        for lang_key in ("aliases_fr", "aliases_en", "aliases_zh"):
            for alias in info.get(lang_key, []):
                if alias.lower() == text_clean:
                    return city_name
    return None

def resolve_airline(text: str, lexique: dict) -> Optional[str]:
    """Résout un nom de compagnie en nom canonique."""
    text_clean = text.strip().lower()

    compagnies = lexique.get("compagnies_aeriennes", {})
    for name, info in compagnies.items():
        if name.lower() == text_clean:
            return name
        if info.get("iata", "").lower() == text_clean:
            return name
        for lang_key in ("aliases_fr", "aliases_en", "aliases_zh"):
            for alias in info.get(lang_key, []):
                if alias.lower() == text_clean:
                    return name
    return None

# ─── Analyse par règles ──────────────────────────────────────────────

def rule_based_nlu(message: str, lexique: dict) -> Optional[Dict]:
    """Pré-analyse par grammaires et règles multilingues."""
    msg = message.lower().strip()

    slots = {}

    # ── Reconnaissance des options du menu BF=7 ──────────────────
    menu_map = {
        "1": ("accepter", {}),
        "2": ("flight", {"cost_relative": "prix_bas"}),
        "3": ("flight_time", {"flight_mod": "premier_disponible"}),
        "4": ("flight_time", {"flight_mod": "dernier_disponible"}),
        "5": ("airline", {}),
        "6": ("demande_details", {}),
        "7": ("flight", {}),
    }
    if msg.strip() in menu_map:
        intent, extra_slots = menu_map[msg.strip()]
        return {"intent": intent, "slots": extra_slots, "confidence": 0.98}

    # Patterns textuels des options
    option_patterns = [
        # 在 option_patterns 列表里加这一条
        (r"(oui.*réserve|je réserve|réserver|book.*it|i.*book|confirm.*book|预订|确认预订)", "accepter", {}),
        (r"(je prends|confirmer|je confirme|yes.*book|i'll take it|i want it|好|确认|就这个|我要)", "accepter", {}),
        (r"(moins cher|cheaper|plus économique|更便宜)", "flight", {"cost_relative": "prix_bas"}),
        (r"(plus tôt|earlier|morning|plus matinal|更早)", "flight_time", {"flight_mod": "premier_disponible"}),
        (r"(plus tard|later|evening|plus tardif|更晚)", "flight_time", {"flight_mod": "dernier_disponible"}),
        (r"(changer.*compagnie|autre compagnie|different airline|换.*航空)", "airline", {}),
        (r"(détails|details|plus d'info|详细)", "demande_details", {}),
        (r"(autre destination|different destination|其他目的地)", "flight", {}),
    ]
    for pattern, intent, extra_slots in option_patterns:
        if re.search(pattern, msg):
            return {"intent": intent, "slots": extra_slots, "confidence": 0.92}


    # Confirmation / Négation (priorité haute)
    if CONFIRMATION_GRAMMAR.match(msg):
        return {"intent": "accepter", "slots": slots, "confidence": 0.97}
    if NEGATION_GRAMMAR.match(msg):
        return {"intent": "rejeter", "slots": slots, "confidence": 0.97}

   # Correction de slot
    if CORRECTION_GRAMMAR.search(msg) and len(msg) < 100:
        # 先提取所有城市
        city_pair = CITY_PAIR_GRAMMAR.search(message)
        if city_pair:
            from_raw = city_pair.group(1).strip()
            to_raw   = city_pair.group(2).strip()
            from_city = resolve_city(from_raw, lexique)
            to_city   = resolve_city(to_raw, lexique)
            if from_city: slots["from_city"] = from_city
            if to_city:   slots["to_city"] = to_city
        # 再单独扫城市名
        villes = lexique.get("villes", {})
        for city_name, info in villes.items():
            all_names = [city_name.lower()] + [a.lower() for a in
                info.get("aliases_fr", []) + info.get("aliases_en", []) + info.get("aliases_zh", [])]
            for name in all_names:
                pattern = r"\b" + re.escape(name) + r"\b" if name.isascii() else re.escape(name)
                if re.search(pattern, msg):
                    if "from_city" not in slots:
                        slots["from_city"] = city_name
                    elif "to_city" not in slots and slots["from_city"] != city_name:
                        slots["to_city"] = city_name
                    break
        # 移除被否定的城市（"pas à Denver" → Denver 不是目的地）
        neg_match = re.search(r"(?:pas\s+à|not|pas)\s+([A-Za-zÀ-ÿ\s]+?)(?:\s|$|,|\.)", msg)
        if neg_match:
            neg_city = resolve_city(neg_match.group(1).strip(), lexique)
            if neg_city and slots.get("to_city") == neg_city:
                slots.pop("to_city", None)
            if neg_city and slots.get("from_city") == neg_city:
                slots.pop("from_city", None)
        return {"intent": "correction_slot", "slots": slots, "confidence": 0.85}

    # Demande de clarification
    clarif = re.search(
        r"(pouvez.vous répéter|je n'ai pas compris|pouvez.vous préciser|qu'est.ce que vous voulez dire"
        r"|can you repeat|i didn't understand|could you clarify"
        r"|能再说一遍|我没听懂|能解释一下)",
        msg
    )
    if clarif:
        return {"intent": "demande_clarification", "slots": slots, "confidence": 0.95}

    # Salutation
    salutations_lex = []
    for lang in ("fr", "en", "zh"):
        salutations_lex.extend(lexique.get("salutations", {}).get("ouverture", {}).get(lang, []))
    for sal in salutations_lex:
        if sal.lower() in msg and len(msg) < 40:
            return {"intent": "salutation", "slots": slots, "confidence": 0.92}

    # Au revoir
    clotures = []
    for lang in ("fr", "en", "zh"):
        clotures.extend(lexique.get("salutations", {}).get("cloture", {}).get(lang, []))
    for bye in clotures:
        if bye.lower() in msg and len(msg) < 50:
            return {"intent": "au_revoir", "slots": slots, "confidence": 0.92}

    # ── Extraction de slots par grammaires ──────────────────────────

    # Paire de villes (de X à Y)
    city_pair = CITY_PAIR_GRAMMAR.search(message)
    if city_pair:
        from_raw = city_pair.group(1).strip()
        to_raw   = city_pair.group(2).strip()
        from_city = resolve_city(from_raw, lexique)
        to_city   = resolve_city(to_raw, lexique)
        if from_city:
            slots["from_city"] = from_city
        if to_city:
            slots["to_city"] = to_city

    # Villes individuelles (recherche dans lexique)
    villes = lexique.get("villes", {})
    for city_name, info in villes.items():
        all_names = [city_name.lower()] + [a.lower() for a in
            info.get("aliases_fr", []) + info.get("aliases_en", []) + info.get("aliases_zh", [])]
        for name in all_names:
            pattern = r"\b" + re.escape(name) + r"\b" if name.isascii() else re.escape(name)
            if re.search(pattern, msg):
                if "from_city" not in slots:
                    slots["from_city"] = city_name
                elif "to_city" not in slots and slots["from_city"] != city_name:
                    slots["to_city"] = city_name
                break

    # Compagnie aérienne
    compagnies = lexique.get("compagnies_aeriennes", {})
    for airline_name, info in compagnies.items():
        all_aliases = [airline_name.lower()] + [a.lower() for a in
            info.get("aliases_fr", []) + info.get("aliases_en", []) + info.get("aliases_zh", [])]
        for alias in all_aliases:
            if alias in msg:
                slots["airline"] = airline_name
                break

    # Classe de voyage
    classes = lexique.get("classes_voyage", {})
    for class_key, class_info in classes.items():
        for lang in ("fr", "en", "zh"):
            for expr in class_info.get(lang, []):
                if expr.lower() in msg:
                    slots["class_type"] = class_key
                    break

    # Type de vol (aller simple / aller-retour / direct / escale)
    type_vol = lexique.get("type_vol", {})
    for tv_key, tv_info in type_vol.items():
        for lang in ("fr", "en", "zh"):
            for expr in tv_info.get(lang, []):
                if expr.lower() in msg:
                    if tv_key in ("aller_simple", "aller_retour"):
                        slots["round_trip"] = tv_key
                    elif tv_key in ("direct", "avec_escale"):
                        slots["flight_stop"] = tv_key
                    break

    # Critères de prix
    price_match = PRICE_GRAMMAR.search(msg)
    if price_match:
        grps = price_match.groups()
        if grps[0]:  # moins de X
            slots["fare_amount"] = int(grps[0])
            slots["cost_relative"] = "budget_max"
        elif grps[2] and grps[3]:  # entre X et Y
            slots["fare_amount"] = int(grps[3])
            slots["cost_relative"] = "budget_range"
        elif grps[4]:  # environ X
            slots["fare_amount"] = int(grps[4])
            slots["cost_relative"] = "budget_approx"
        elif grps[5]:  # X dollars
            slots["fare_amount"] = int(grps[5])

    # Termes évaluatifs (pas cher, direct, le plus tôt...)
    termes_eval = lexique.get("termes_evaluatifs", {})
    for eval_key, eval_info in termes_eval.items():
        for lang in ("fr", "en", "zh"):
            for term in eval_info.get(lang, []):
                if term.lower() in msg:
                    slots["cost_relative"] = eval_key
                    break

    # Nombre de passagers
    pax_match = PASSENGER_GRAMMAR.search(msg)
    if pax_match:
        n = int(next(g for g in pax_match.groups() if g is not None))
        slots["passenger_count"] = n

    # Date relative
    date_rel = DATE_RELATIVE_GRAMMAR.search(msg)
    if date_rel:
        slots["depart_date_relative"] = date_rel.group(0)

    # Date absolue
    date_abs = DATE_GRAMMAR.search(msg)
    if date_abs and "depart_date_relative" not in slots:
        d = date_abs.groupdict()
        day = d.get("day") or d.get("day2") or d.get("day3") or d.get("cn_day")
        month = (d.get("month_num") or d.get("month_fr") or
                 d.get("month_en") or d.get("cn_month"))
        if day:
            slots["depart_date_day_number"] = int(day)
        if month:
            slots["depart_month"] = month

    # Heure de départ
    time_match = TIME_GRAMMAR.search(msg)
    if time_match:
        d = time_match.groupdict()
        h = d.get("h1") or d.get("h2") or d.get("h3") or d.get("h4") or d.get("h5")
        m = d.get("m1") or d.get("m2") or d.get("m3") or d.get("m5") or "00"
        ampm = d.get("ampm", "")
        if h:
            hh = int(h)
            if ampm and ampm.lower() == "pm" and hh < 12:
                hh += 12
            slots["depart_time"] = f"{hh:02d}:{m.zfill(2) if m else '00'}"

    # Période de la journée
    period_match = PERIOD_GRAMMAR.search(msg)
    if period_match and "depart_time" not in slots:
        period_raw = period_match.group(1).lower()
        period_map = {
            "tôt le matin": "matin", "matin": "matin", "early morning": "matin",
            "morning": "matin", "清晨": "matin", "早上": "matin", "早间": "matin", "上午": "matin",
            "après-midi": "après-midi", "afternoon": "après-midi", "下午": "après-midi",
            "fin d'après-midi": "après-midi", "late afternoon": "après-midi",
            "soir": "soir", "evening": "soir", "晚上": "soir",
            "nuit": "nuit", "night": "nuit", "夜间": "nuit",
            "midi": "midi", "noon": "midi", "中午": "midi", "early": "matin",
        }
        slots["depart_period"] = period_map.get(period_raw, period_raw)

    # Modificateur de vol (le plus tôt, le plus tard, direct...)
    mods_fr = lexique.get("termes_evaluatifs", {})
    for mod_key in ("premier_disponible", "dernier_disponible", "rapidite"):
        mod_info = mods_fr.get(mod_key, {})
        for lang in ("fr", "en", "zh"):
            for term in mod_info.get(lang, []):
                if term.lower() in msg:
                    slots["flight_mod"] = mod_key
                    break

    # ── Détermination de l'intent ────────────────────────────────────

    # Transport terrestre
    ground = re.search(
        r"(navette|shuttle|taxi|limousine|bus|métro|centre.ville|downtown|如何到|机场大巴|豪华轿车)",
        msg
    )
    if ground:
        return {"intent": "ground_service", "slots": slots, "confidence": 0.88}

    # Type d'appareil
    if re.search(r"(quel.*avion|type.*appareil|what.*aircraft|what.*plane|飞机型号|什么飞机)", msg):
        return {"intent": "aircraft", "slots": slots, "confidence": 0.88}

    # Info compagnie
    if re.search(r"(quelle.*compagnie|which.*airline|哪些航空|vols.*compagnie)", msg) and not slots.get("from_city"):
        return {"intent": "airline", "slots": slots, "confidence": 0.85}

    # Horaires
    if re.search(r"(à quelle heure|horaire|quand.*vol|what time|when does|flight schedule|几点|时刻表|什么时候)", msg):
        return {"intent": "flight_time", "slots": slots, "confidence": 0.88}

    # Prix / tarif
    if re.search(r"(combien.*coûte|quel.*prix|quel.*tarif|how much|price|fare|cost|票价|多少钱|价格)", msg):
        return {"intent": "airfare", "slots": slots, "confidence": 0.88}

    # Recherche de vol (intent principal)
    if slots.get("from_city") or slots.get("to_city"):
        return {"intent": "flight", "slots": slots, "confidence": 0.85}

    if re.search(
        r"(vol|flight|billet|ticket|voyage|voyager|partir|réserver|cherche|veux|voudrais"
        r"|book|travel|fly|want|need|find"
        r"|航班|机票|飞|想去|预订|查询)", msg
    ):
        return {"intent": "flight", "slots": slots, "confidence": 0.75}

    if slots:
        return {"intent": "flight", "slots": slots, "confidence": 0.70}

    return None

# ─── Module NLU LLM ──────────────────────────────────────────────────

def llm_nlu(message: str, context: str) -> Dict:
    """Utilise le LLM pour classifier l'intent et extraire les slots."""
    slots_desc = ", ".join(SLOT_NAMES)
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"You are an NLU module for a multilingual flight booking chatbot (VoyageAssistant).\n"
        f"The user may write in French, English, or Chinese.\n"
        f"Analyze the user's message and return ONLY a valid JSON with:\n"
        f"- \"intent\": one of {INTENT_LIST}\n"
        f"- \"slots\": dict with possible keys [{slots_desc}] (only detected ones)\n"
        f"- \"langue_detectee\": \"fr\", \"en\", or \"zh\"\n"
        f"- \"confiance\": float 0-1\n"
        f"\nFor city names, return the canonical English name (e.g. 'Paris', 'New York', 'Tokyo').\n"
        f"For airlines, return the full name (e.g. 'Delta Air Lines', 'Air France').\n"
        f"For class_type use: 'économique', 'affaires', or 'première'.\n"
        f"For round_trip use: 'aller_simple' or 'aller_retour'.\n"
        f"For flight_stop use: 'direct' or 'avec_escale'.\n"
        f"IMPORTANT: Return ONLY the JSON, no text before or after.\n"
        f"<|eot_id|>\n"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"Dialogue context: {context}\n"
        f"User message: \"{message}\"\n"
        f"<|eot_id|>\n"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    try:
        result = llm_pipeline(prompt, max_new_tokens=300, temperature=0.1, do_sample=True)
        generated = result[0]["generated_text"]
        assistant_part = generated.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        assistant_part = assistant_part.replace("<|eot_id|>", "").strip()
        json_match = re.search(r"\{[^{}]*\}", assistant_part, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            if "intent" in parsed and parsed["intent"] in INTENT_LIST:
                parsed.setdefault("slots", {})
                parsed.setdefault("confiance", 0.7)
                return parsed
    except Exception as e:
        print(f"[NLU LLM Error] {e}")
    return {"intent": "flight", "slots": {}, "confiance": 0.3}

# Mots-clés liés aux vols
FLIGHT_KEYWORDS = re.compile(
    r"(vol|flight|billet|ticket|avion|plane|aéroport|airport|compagnie|airline"
    r"|départ|arrivée|departure|arrival|réserver|book|voyage|travel"
    r"|航班|机票|飞机|机场|航空|出发|到达|预订|旅行)",
    re.IGNORECASE
)

def analyze_user_input(message: str, context: str, lexique: dict) -> Dict:
    """Pipeline NLU : détection langue → règles → LLM."""
    lang = detect_language(message)
    msg_lower = message.lower().strip()
    is_question = any(msg_lower.endswith(c) for c in ("?", "？"))
    has_flight_kw = bool(FLIGHT_KEYWORDS.search(msg_lower))

    rule_result = rule_based_nlu(message, lexique)

    # ── Out-of-scope : priorité haute, avant le retour anticipé ──────
    has_city_only = rule_result and set(rule_result.get("slots", {}).keys()) <= {"from_city", "to_city"}
    if is_question and not has_flight_kw and (not rule_result or has_city_only):
        return {"intent": "out_of_scope", "slots": {}, "confidence": 0.85, "lang": lang}

    if rule_result and rule_result["confidence"] >= 0.85:
        rule_result["lang"] = lang
        return rule_result

    llm_result = llm_nlu(message, context)

    # Fusion : les slots des règles priment sur le LLM
    if rule_result and rule_result.get("slots"):
        merged = {**llm_result.get("slots", {}), **rule_result["slots"]}
        llm_result["slots"] = merged

    llm_result["lang"] = lang
    return llm_result

print("✓ Module NLU multilingue chargé (FR/EN/ZH)")
print(f"  Grammaires actives : DATE, TIME, DATE_RELATIVE, CONFIRMATION, NEGATION,")
print(f"  PRICE, PASSENGER, CORRECTION, CITY_PAIR, PERIOD")


✓ Module NLU multilingue chargé (FR/EN/ZH)
  Grammaires actives : DATE, TIME, DATE_RELATIVE, CONFIRMATION, NEGATION,
  PRICE, PASSENGER, CORRECTION, CITY_PAIR, PERIOD


## 6. Moteur de recherche de vols

Le moteur filtre la base de 300 vols selon les critères extraits par le NLU.  
Un système de score pondéré classe les résultats par pertinence.


In [127]:
def normalize_city(city: str) -> str:
    return strip_accents(city.lower().strip())

def score_flight(flight: dict, slots: dict, lexique: dict) -> float:
    """Calcule un score de pertinence pour un vol selon les slots remplis."""
    score = 0.0

    # Ville de départ (obligatoire si présent)
    if "from_city" in slots:
        if normalize_city(flight.get("from_city", "")) == normalize_city(slots["from_city"]):
            score += 5.0
        else:
            score -= 10.0  # éliminatoire

    # Ville d'arrivée (obligatoire si présent)
    if "to_city" in slots:
        if normalize_city(flight.get("to_city", "")) == normalize_city(slots["to_city"]):
            score += 5.0
        else:
            score -= 10.0  # éliminatoire

    # Compagnie aérienne
    if "airline" in slots:
        if normalize_city(flight.get("airline", "")) == normalize_city(slots["airline"]):
            score += 3.0

    # Vol direct / avec escale
    if "flight_stop" in slots:
        if slots["flight_stop"] == "direct" and flight.get("stops", 1) == 0:
            score += 3.0
        elif slots["flight_stop"] == "avec_escale" and flight.get("stops", 0) > 0:
            score += 2.0
        elif slots["flight_stop"] == "direct" and flight.get("stops", 0) > 0:
            score -= 4.0

    # Budget
    if "fare_amount" in slots:
        price = flight.get("price", 999)
        if slots.get("cost_relative") == "budget_max":
            if price <= slots["fare_amount"]:
                score += 3.0
            else:
                score -= 3.0
        elif slots.get("cost_relative") in ("prix_bas", "budget_approx"):
            diff = abs(price - slots["fare_amount"])
            score += max(0, 3.0 - diff / 100)
    elif "cost_relative" in slots:
        cr = slots["cost_relative"]
        price = flight.get("price", 200)
        if cr in ("prix_bas", "tres_bon_marche", "bon_marche"):
            score += (300 - price) / 100  # favorise prix bas
        elif cr == "rapidite":
            score += (300 - flight.get("duration_minutes", 300)) / 100

    # Période de départ
    if "depart_period" in slots:
        depart_h = int(flight.get("depart_time", "12:00").split(":")[0])
        period = slots["depart_period"]
        periodes = lexique.get("expressions_temporelles", {}).get("periodes_journee", {}).get("fr", {})
        p_info = periodes.get(period, {})
        if p_info:
            h_start = int(p_info.get("heure_debut", "00:00").split(":")[0])
            h_end   = int(p_info.get("heure_fin",   "23:00").split(":")[0])
            if h_start <= depart_h < h_end:
                score += 2.0
            else:
                score -= 1.0

    # Heure de départ précise
    if "depart_time" in slots:
        try:
            wanted_h, wanted_m = map(int, slots["depart_time"].split(":"))
            flight_h, flight_m = map(int, flight.get("depart_time", "12:00").split(":"))
            diff_min = abs((wanted_h * 60 + wanted_m) - (flight_h * 60 + flight_m))
            score += max(0, 3.0 - diff_min / 60)
        except:
            pass

    # Modificateur (le plus tôt / le plus tard / le plus rapide)
    if "flight_mod" in slots:
        mod = slots["flight_mod"]
        if mod == "premier_disponible":
            depart_h = int(flight.get("depart_time", "12:00").split(":")[0])
            score += (24 - depart_h) * 0.1  # favorise les vols tôt
        elif mod == "dernier_disponible":
            depart_h = int(flight.get("depart_time", "12:00").split(":")[0])
            score += depart_h * 0.1  # favorise les vols tard
        elif mod == "rapidite":
            score += (300 - flight.get("duration_minutes", 300)) / 100

    return score


def search_flights(slots: dict, flights_db: list, lexique: dict,
                   exclude_ids: set = None, top_n: int = 5) -> List[dict]:
    """Retourne les top_n vols les plus pertinents selon les critères."""
    if exclude_ids is None:
        exclude_ids = set()

    candidates = [f for f in flights_db if f.get("flight_id") not in exclude_ids]

    if not slots:
        return random.sample(candidates, min(top_n, len(candidates)))

    scored = [(f, score_flight(f, slots, lexique)) for f in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)

    results = [(f, sc) for f, sc in scored if sc > 0]
    return [f for f, _ in results[:top_n]]


def get_arrive_date(depart_date: str, depart_time: str, arrive_time: str) -> str:
    """如果到达时间 < 出发时间，说明跨夜，日期+1天。"""
    from datetime import date, timedelta
    if not depart_date:
        return "—"
    d_h = int(depart_time.split(':')[0])
    a_h = int(arrive_time.split(':')[0])
    d = date.fromisoformat(depart_date)
    if a_h < d_h:  # 跨夜
        d = d + timedelta(days=1)
    return d.isoformat()


def format_flight_short(flight: dict, lang: str = "fr") -> str:
    """Affichage court d'un vol (pour recommandation)."""
    stops_label = {
        "fr": {0: "direct ✈️", 1: "1 escale 🔄"},
        "en": {0: "nonstop ✈️", 1: "1 stop 🔄"},
        "zh": {0: "直飞 ✈️", 1: "1次中转 🔄"},
    }
    stops = flight.get("stops", 0)
    stops_str = stops_label.get(lang, stops_label["fr"]).get(stops, f"{stops} escales")
    dur = flight.get("duration_minutes", 0)
    dur_str = f"{dur // 60}h{dur % 60:02d}"
    depart_date = flight.get("depart_date", "—")
    arrive_date = get_arrive_date(depart_date, flight["depart_time"], flight["arrive_time"])

    if lang == "fr":
        return (
            f"**{flight['flight_id']}** — {flight['airline']}\n"
            f"🛫 {flight['from_city']} ({flight['from_iata']}) → {flight['to_city']} ({flight['to_iata']})\n"
            f"🗓 {depart_date} 🕐 Départ {flight['depart_time']} · Arrivée {flight['arrive_time']} ({arrive_date}) · {dur_str} · {stops_str}\n"
            f"✈️ {flight.get('aircraft', '?')} · 💰 {flight['price']}$"
        )
    elif lang == "en":
        return (
            f"**{flight['flight_id']}** — {flight['airline']}\n"
            f"🛫 {flight['from_city']} ({flight['from_iata']}) → {flight['to_city']} ({flight['to_iata']})\n"
            f"🗓 {depart_date} 🕐 Dep {flight['depart_time']} · Arr {flight['arrive_time']} ({arrive_date}) · {dur_str} · {stops_str}\n"
            f"✈️ {flight.get('aircraft', '?')} · 💰 ${flight['price']}"
        )
    else:  # zh
        return (
            f"**{flight['flight_id']}** — {flight['airline']}\n"
            f"🛫 {flight['from_city']} ({flight['from_iata']}) → {flight['to_city']} ({flight['to_iata']})\n"
            f"🗓 {depart_date} 🕐 出发 {flight['depart_time']} · 到达 {flight['arrive_time']} ({arrive_date}) · {dur_str} · {stops_str}\n"
            f"✈️ {flight.get('aircraft', '?')} · 💰 {flight['price']}美元"
        )




def format_flight_details(flight: dict, lang: str = "fr") -> str:
    """Affichage détaillé d'un vol."""
    dur = flight.get("duration_minutes", 0)
    dur_str = f"{dur // 60}h{dur % 60:02d}"
    stops = flight.get("stops", 0)
    depart_date = flight.get("depart_date", "—")
    arrive_date = get_arrive_date(depart_date, flight["depart_time"], flight["arrive_time"])

    if lang == "fr":
        escale_str = "Vol direct" if stops == 0 else f"{stops} escale(s)"
        return (
            f"✈️ **Vol {flight['flight_id']}** — {flight['airline']}\n"
            f"📍 **Départ** : {flight['from_city']} ({flight['from_iata']})\n"
            f"📍 **Arrivée** : {flight['to_city']} ({flight['to_iata']})\n"
            f"🗓 **Date départ** : {depart_date}\n"
            f"🕐 **Heure départ** : {flight['depart_time']}\n"
            f"🗓 **Date arrivée** : {arrive_date}\n"
            f"🕐 **Heure arrivée** : {flight['arrive_time']}\n"
            f"⏱ **Durée** : {dur_str}\n"
            f"🔄 **Escales** : {escale_str}\n"
            f"🛩 **Appareil** : {flight.get('aircraft', 'N/A')}\n"
            f"💰 **Prix** : {flight['price']}$"
        )
    elif lang == "en":
        stop_str = "Nonstop" if stops == 0 else f"{stops} stop(s)"
        return (
            f"✈️ **Flight {flight['flight_id']}** — {flight['airline']}\n"
            f"📍 **From** : {flight['from_city']} ({flight['from_iata']})\n"
            f"📍 **To** : {flight['to_city']} ({flight['to_iata']})\n"
            f"🗓 **Departure date** : {depart_date}\n"
            f"🕐 **Departure** : {flight['depart_time']}\n"
            f"🗓 **Arrival date** : {arrive_date}\n"
            f"🕐 **Arrival** : {flight['arrive_time']}\n"
            f"⏱ **Duration** : {dur_str}\n"
            f"🔄 **Stops** : {stop_str}\n"
            f"🛩 **Aircraft** : {flight.get('aircraft', 'N/A')}\n"
            f"💰 **Price** : ${flight['price']}"
        )
    else:  # zh
        stop_str = "直飞" if stops == 0 else f"{stops}次中转"
        return (
            f"✈️ **航班 {flight['flight_id']}** — {flight['airline']}\n"
            f"📍 **出发** : {flight['from_city']} ({flight['from_iata']})\n"
            f"📍 **到达** : {flight['to_city']} ({flight['to_iata']})\n"
            f"🗓 **出发日期** : {depart_date}\n"
            f"🕐 **出发时间** : {flight['depart_time']}\n"
            f"🗓 **到达日期** : {arrive_date}\n"
            f"🕐 **到达时间** : {flight['arrive_time']}\n"
            f"⏱ **飞行时长** : {dur_str}\n"
            f"🔄 **中转** : {stop_str}\n"
            f"🛩 **机型** : {flight.get('aircraft', 'N/A')}\n"
            f"💰 **价格** : {flight['price']}美元"
        )

print("✓ Moteur de recherche de vols chargé (300 vols)")


✓ Moteur de recherche de vols chargé (300 vols)


## 7. Module NLG — Génération de langage naturel (multilingue)

Le module NLG (Natural Language Generation) produit des réponses variées et expressives via :
1. **Templates** statiques (pour salutation, au revoir — rapides et fiables)
2. **LLM** (Llama-3.2-3B-Instruct) pour les réponses contextuelles avec contenu émotionnel

### Contenu émotionnel (étape 5)
Le prompt NLG inclut des instructions émotionnelles adaptées à l'état détecté :
enthousiaste, rassurant, empathique, guidant, efficace, célébrant.


In [128]:
# Templates statiques (salutation, au revoir, fallback)
RESPONSE_TEMPLATES = {
    "fr": {
        "salutation": [
            "Bonjour ! Je suis VoyageAssistant, votre expert en vols. ✈️ Comment puis-je vous aider aujourd'hui ?",
            "Salut ! Prêt à décoller ? Dites-moi où vous souhaitez aller ! 🌍",
            "Bonsoir ! Je suis là pour trouver le vol idéal pour vous. Par où commençons-nous ? 🛫",
        ],
        "au_revoir": [
            "Merci d'avoir utilisé VoyageAssistant ! ✈️ À bientôt pour votre prochain voyage !",
            "Bon voyage ! N'hésitez pas à revenir si vous avez besoin d'aide. 🌍",
            "À bientôt ! Que votre voyage soit mémorable ! 🎉",
        ],
        "incomprehension": [
            "Je suis spécialisé dans la réservation de vols. Pouvez-vous me dire d'où vous partez et où vous souhaitez aller ?",
            "Je n'ai pas bien saisi. Parlez-moi de votre voyage : ville de départ, destination, date... ✈️",
        ],
        "demande_depart": [
            "Depuis quelle ville souhaitez-vous partir ?",
            "Quelle est votre ville de départ ? 🛫",
        ],
        "demande_arrivee": [
            "Et quelle est votre destination ?",
            "Où souhaitez-vous vous rendre ? 🌍",
        ],
        "confirmation": [
            "Je confirme votre réservation. Voici les détails :",
            "Parfait ! Réservation en cours... ✅",
        ],
    },
    "en": {
        "salutation": [
            "Hello! I'm VoyageAssistant, your flight expert. ✈️ How can I help you today?",
            "Hi there! Ready to take off? Tell me where you want to go! 🌍",
            "Good evening! I'm here to find the perfect flight for you. Where shall we start? 🛫",
        ],
        "au_revoir": [
            "Thank you for using VoyageAssistant! ✈️ See you for your next trip!",
            "Have a great trip! Feel free to come back if you need help. 🌍",
            "Goodbye! May your journey be unforgettable! 🎉",
        ],
        "incomprehension": [
            "I specialize in flight booking. Could you tell me your departure and destination cities?",
            "I didn't quite catch that. Tell me about your trip: departure city, destination, date... ✈️",
        ],
        "demande_depart": [
            "Which city are you departing from?",
            "What is your departure city? 🛫",
        ],
        "demande_arrivee": [
            "And what is your destination?",
            "Where would you like to go? 🌍",
        ],
        "confirmation": [
            "I confirm your booking. Here are the details:",
            "Perfect! Booking in progress... ✅",
        ],
    },
    "zh": {
        "salutation": [
            "您好！我是VoyageAssistant，您的专属航班助手。✈️ 今天我能为您做什么？",
            "嗨！准备起飞了吗？告诉我您想去哪里！🌍",
            "晚上好！我来帮您找到最完美的航班。我们从哪里开始？🛫",
        ],
        "au_revoir": [
            "感谢使用VoyageAssistant！✈️ 期待您下次的旅程！",
            "祝旅途愉快！如果需要帮助随时回来。🌍",
            "再见！愿您的旅程难忘！🎉",
        ],
        "incomprehension": [
            "我专门负责航班预订。请问您从哪里出发，想去哪里？",
            "我没有完全理解。请告诉我您的旅行信息：出发城市、目的地、日期……✈️",
        ],
        "demande_depart": [
            "请问您从哪个城市出发？",
            "您的出发城市是哪里？🛫",
        ],
        "demande_arrivee": [
            "您想去哪里？",
            "您的目的地是哪里？🌍",
        ],
        "confirmation": [
            "我确认您的预订。以下是详细信息：",
            "完美！预订进行中……✅",
        ],
    },
}

# Styles émotionnels pour le prompt NLG
EMOTIONAL_STYLES = {
    "enthousiaste": {
        "fr": "Sois enthousiaste et chaleureux. Use des emojis (✈️ 🌍 🎉). Transmets de l'excitation pour le voyage.",
        "en": "Be enthusiastic and warm. Use emojis (✈️ 🌍 🎉). Convey excitement about the trip.",
        "zh": "热情活泼，使用表情符号（✈️ 🌍 🎉），传递旅行的兴奋感。",
    },
    "rassurant": {
        "fr": "Sois calme et bienveillant. Fournis des détails précis pour rassurer. Ton empathique.",
        "en": "Be calm and caring. Provide precise details to reassure. Empathetic tone.",
        "zh": "语气平静关怀，提供详细信息以安抚用户，富有同理心。",
    },
    "empathique": {
        "fr": "Reconnais la frustration de l'utilisateur. Propose une solution concrète rapidement.",
        "en": "Acknowledge the user's frustration. Quickly propose a concrete solution.",
        "zh": "理解用户的困扰，迅速提供具体解决方案。",
    },
    "guidant": {
        "fr": "Guide l'utilisateur avec des options numérotées claires (facteur de branchement 5-8). Pose des questions ciblées.",
        "en": "Guide the user with clear numbered options (branching factor 5-8). Ask targeted questions.",
        "zh": "用编号选项引导用户（分支因子5-8），提出针对性问题。",
    },
    "efficace": {
        "fr": "Sois direct et concis. Réponse courte, va à l'essentiel, pas de fioritures.",
        "en": "Be direct and concise. Short answer, get to the point, no frills.",
        "zh": "简洁直接，回答简短，直奔主题。",
    },
    "celebrant": {
        "fr": "Félicite l'utilisateur pour sa réservation ! Ton festif, emojis 🎉 🥳 ✅.",
        "en": "Congratulate the user on their booking! Festive tone, emojis 🎉 🥳 ✅.",
        "zh": "恭喜用户完成预订！节日气氛，使用表情符号🎉🥳✅。",
    },
    "neutre": {
        "fr": "Ton informatif standard. Présente les informations clairement.",
        "en": "Standard informative tone. Present information clearly.",
        "zh": "标准信息化语气，清晰呈现信息。",
    },
}

def detect_emotional_state(message: str, intent: str, state: str) -> str:
    """Détermine le style émotionnel approprié."""
    msg = message.lower() if message else ""
    # Utilisateur frustré
    if re.search(r"(trop cher|nul|pfff|rien|ugh|too expensive|太贵|找不到|烦)", msg):
        return "empathique"
    # Utilisateur pressé
    if re.search(r"(urgent|vite|dépêchez|asap|马上|赶紧|快)", msg):
        return "efficace"
    # Utilisateur enthousiaste
    if re.search(r"(super|génial|j'adore|amazing|fantastic|太棒了|好期待)", msg):
        return "enthousiaste"
    # Utilisateur hésitant
    if re.search(r"(peut-être|je sais pas|ça dépend|hmm|maybe|也许|不知道)", msg):
        return "guidant"
    # Confirmation de réservation
    if state == "BOOKING_CONFIRMED":
        return "celebrant"
    # Présentation de résultats
    if intent in ("flight", "airfare", "flight_time") and state == "RECOMMENDATION":
        return "enthousiaste"
    # Rejet multiple
    if state == "FOLLOW_UP" and intent == "rejeter":
        return "empathique"
    return "neutre"


def llm_generate_response(intent: str, slots: dict, flight_data: dict,
                           history: str, state: str, lang: str = "fr",
                           user_message: str = "") -> Optional[str]:
    """Utilise le LLM pour générer une réponse contextuelle avec contenu émotionnel."""
    emotional_style = detect_emotional_state(user_message, intent, state)
    style_instr = EMOTIONAL_STYLES.get(emotional_style, EMOTIONAL_STYLES["neutre"]).get(lang, "")

    flight_info = ""
    if flight_data:
        stops_str = "VOL DIRECT (0 escale)" if flight_data.get("stops", 0) == 0 else f"{flight_data['stops']} escale(s)"
        flight_info = (
            f"DONNÉES EXACTES DU VOL — NE PAS MODIFIER :\n"
            f"- Numéro de vol : {flight_data['flight_id']}\n"
            f"- Compagnie : {flight_data['airline']}\n"
            f"- Départ : {flight_data['from_city']} ({flight_data['from_iata']})\n"
            f"- Arrivée : {flight_data['to_city']} ({flight_data['to_iata']})\n"
            f"- Heure départ : {flight_data['depart_time']}\n"
            f"- Heure arrivée : {flight_data['arrive_time']}\n"
            f"- Escales : {stops_str}\n"
            f"- Durée totale vol : {flight_data.get('duration_minutes', 0)} min\n"
            f"- Appareil : {flight_data.get('aircraft', 'N/A')}\n"
            f"- Prix : {flight_data['price']}$\n"
            f"INTERDICTION ABSOLUE : Ne jamais inventer ou modifier ces données. "
            f"La compagnie est UNIQUEMENT {flight_data['airline']}, pas une autre.\n"
        )

    lang_names = {"fr": "French", "en": "English", "zh": "Chinese"}
    lang_name = lang_names.get(lang, "French")

    system_prompts = {
        "fr": (
            "Tu es VoyageAssistant, un assistant expert en réservation de vols, "
            "chaleureux, professionnel et multilingue.\n"
            "Règles :\n"
            "- Réponds UNIQUEMENT en français\n"
            "- Sois concis (2-4 phrases maximum)\n"
            f"- Style émotionnel : {style_instr}\n"
            "- Mentionne le numéro de vol et la compagnie si un vol est disponible\n"
            "- Pose une question de suivi pour maintenir la conversation\n"
            "- Ne répète jamais une réponse précédente\n"
            "- Ne mentionne que les données fournies, n'invente rien"
        ),
        "en": (
            "You are VoyageAssistant, an expert flight booking assistant, "
            "warm, professional, and multilingual.\n"
            "Rules:\n"
            "- Respond ONLY in English\n"
            "- Be concise (2-4 sentences maximum)\n"
            f"- Emotional style: {style_instr}\n"
            "- Mention flight number and airline if a flight is available\n"
            "- Ask a follow-up question to maintain the conversation\n"
            "- Never repeat a previous response\n"
            "- Only mention provided data, never invent anything"
        ),
        "zh": (
            "你是VoyageAssistant，一个专业、热情的多语言航班预订助手。\n"
            "规则：\n"
            "- 只用中文回复\n"
            "- 简洁（最多2-4句话）\n"
            f"- 情感风格：{style_instr}\n"
            "- 如果有航班信息，提及航班号和航空公司\n"
            "- 提出后续问题以保持对话\n"
            "- 绝不重复之前的回复\n"
            "- 只提及提供的数据，不要凭空捏造"
        ),
    }
    system_body = system_prompts.get(lang, system_prompts["fr"])

    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"CRITICAL RULE: You MUST respond ONLY in {lang_name}. Never switch languages.\n\n"
        f"{system_body}<|eot_id|>\n"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"Dialogue state: {state}\n"
        f"Detected intent: {intent}\n"
        f"Collected slots: {json.dumps(slots, ensure_ascii=False)}\n"
        f"{flight_info}"
        f"Recent history: {history[-600:] if history else 'Start of conversation'}\n"
        f"User last message: {user_message}\n"
        f"Generate VoyageAssistant response:<|eot_id|>\n"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    try:
        result = llm_pipeline(prompt, max_new_tokens=300, temperature=0.75, do_sample=True)
        generated = result[0]["generated_text"]
        response = generated.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.replace("<|eot_id|>", "").strip()
        response = response.split("<|")[0].strip()
        if len(response) > 15:
            return response
    except Exception as e:
        print(f"[NLG LLM Error] {e}")
    return None


def generate_response(intent: str, slots: dict, flight: dict,
                      history: str, state: str, lang: str = "fr",
                      user_message: str = "") -> str:
    """Génère une réponse : templates pour les cas simples, LLM pour le reste."""
    # Templates directs pour salutation / au revoir
    if intent in ("salutation", "au_revoir"):
        tpl = RESPONSE_TEMPLATES.get(lang, RESPONSE_TEMPLATES["fr"])
        return random.choice(tpl.get(intent, tpl["incomprehension"]))

    # LLM pour les réponses contextuelles
    llm_resp = llm_generate_response(intent, slots, flight, history, state, lang, user_message)
    if llm_resp:
        # Append formatted flight if not mentioned
        if flight and flight.get("flight_id") not in llm_resp:
            llm_resp += "\n\n" + format_flight_short(flight, lang)
        return llm_resp

    # Fallback : template + vol formaté
    tpl = RESPONSE_TEMPLATES.get(lang, RESPONSE_TEMPLATES["fr"])
    tpl_key = intent if intent in tpl else "incomprehension"
    resp = random.choice(tpl.get(tpl_key, tpl["incomprehension"]))
    if flight:
        resp += "\n\n" + format_flight_short(flight, lang)
    return resp

print("✓ Module NLG multilingue chargé (FR/EN/ZH)")
print(f"  Styles émotionnels : {list(EMOTIONAL_STYLES.keys())}")


✓ Module NLG multilingue chargé (FR/EN/ZH)
  Styles émotionnels : ['enthousiaste', 'rassurant', 'empathique', 'guidant', 'efficace', 'celebrant', 'neutre']


## 8. Gestionnaire de dialogue

Le `DialogManager` orchestre le système via une machine à états.  
Il gère les transitions, accumule les slots, et coordonne NLU → Moteur → NLG.

États : `INIT → GREETING → COLLECTING → RECOMMENDATION → DETAILS → BOOKING → FOLLOW_UP → GOODBYE`


In [129]:
class DialogManager:
    """Gestionnaire de dialogue à états pour VoyageAssistant."""

    STATES = ["INIT", "GREETING", "COLLECTING", "RECOMMENDATION",
              "DETAILS", "BOOKING", "BOOKING_CONFIRMED", "FOLLOW_UP", "GOODBYE"]

    NO_RESULTS = {
        "fr": "Je n'ai pas trouvé de vol correspondant à vos critères. 😔 Voulez-vous modifier vos préférences ? (date, compagnie, budget...)",
        "en": "I couldn't find a flight matching your criteria. 😔 Would you like to adjust your preferences? (date, airline, budget...)",
        "zh": "很遗憾，没有找到符合您要求的航班。😔 需要调整一下条件吗？（日期、航空公司、价格……）",
    }

    MISSING_FROM = {
        "fr": "Depuis quelle ville souhaitez-vous partir ? 🛫",
        "en": "Which city are you departing from? 🛫",
        "zh": "请问您从哪个城市出发？🛫",
    }

    MISSING_TO = {
        "fr": "Et quelle est votre destination ? 🌍",
        "en": "And what is your destination? 🌍",
        "zh": "您想去哪里？🌍",
    }

    BOOKING_CONFIRMED_MSG = {
        "fr": "🎉 Réservation confirmée ! Vol **{fid}** {airline} {from_city} → {to_city}, départ {depart}. Bon voyage ! ✈️",
        "en": "🎉 Booking confirmed! Flight **{fid}** {airline} {from_city} → {to_city}, departure {depart}. Have a great trip! ✈️",
        "zh": "🎉 预订成功！航班 **{fid}** {airline} {from_city} → {to_city}，出发时间 {depart}。祝旅途愉快！✈️",
    }

    OUT_OF_SCOPE = {
        "fr": "Je suis spécialisé dans la réservation de vols. Je ne peux pas vous aider pour cela, mais je peux vous trouver le meilleur vol ! Où souhaitez-vous aller ? ✈️",
        "en": "I specialize in flight booking. I can't help with that, but I can find you the best flight! Where would you like to go? ✈️",
        "zh": "我专门负责航班预订，无法帮您处理这个问题，但我可以帮您找到最好的航班！您想去哪里？✈️",
    }

    CLARIFICATION_ANSWERS = {
        "fr": "Bien sûr ! Je vais vous répéter les informations. Avez-vous d'autres questions ?",
        "en": "Of course! Let me repeat that. Do you have other questions?",
        "zh": "当然！让我再说一遍。您还有其他问题吗？",
    }

    # ── BUG FIX : messages quand il n'y a plus d'alternative dans ce créneau
    NO_EARLIER = {
        "fr": "Il n'y a pas de vol plus tôt sur cette route. 😔 Voici le premier vol disponible :",
        "en": "No earlier flight available on this route. 😔 Here's the earliest available:",
        "zh": "这条航线没有更早的航班了。😔 以下是最早的可用航班：",
    }
    NO_LATER = {
        "fr": "Il n'y a pas de vol plus tard sur cette route. 😔 Voici le dernier vol disponible :",
        "en": "No later flight available on this route. 😔 Here's the latest available:",
        "zh": "这条航线没有更晚的航班了。😔 以下是最晚的可用航班：",
    }

    def __init__(self, flights_db, lexique):
        self.flights_db = flights_db
        self.lexique = lexique
        self.reset()

    def reset(self):
        self.state = "INIT"
        self.slots = {}
        self.lang = "fr"
        self.excluded_ids = set()
        self.current_flight = None
        self.history = []
        self.turn_count = 0
        self.last_results = []

    def get_context_str(self) -> str:
        last_turns = self.history[-6:]
        return " | ".join(
            f"{'USR' if t['role'] == 'user' else 'BOT'}: {t['message'][:80]}"
            for t in last_turns
        )

    def get_conversation_log(self) -> dict:
        return {
            "timestamp": datetime.now().isoformat(),
            "lang": self.lang,
            "turns": self.turn_count,
            "final_state": self.state,
            "slots_collected": self.slots,
            # ── BUG FIX : utilise current_flight si non None, sinon None
            "flight_booked": self.current_flight.get("flight_id") if self.current_flight else None,
            "history": self.history,
        }

    def update_slots(self, new_slots: dict):
        for k, v in new_slots.items():
            if v is not None and not k.startswith("_"):
                self.slots[k] = v

    def _gen(self, intent, flight=None, user_msg=""):
        ctx = self.get_context_str()
        return generate_response(intent, self.slots, flight, ctx,
                                 self.state, self.lang, user_msg)

    def _do_search(self) -> List[dict]:
        results = search_flights(
            self.slots, self.flights_db, self.lexique,
            exclude_ids=self.excluded_ids, top_n=7
        )
        self.last_results = results
        return results

    def process(self, user_message: str) -> str:
        self.turn_count += 1
        self.history.append({"role": "user", "message": user_message})

        ctx = self.get_context_str()
        nlu = analyze_user_input(user_message, ctx, self.lexique)
        intent = nlu["intent"]
        detected_lang = nlu.get("lang", self.lang)
        if not user_message.strip().isdigit():
            self.lang = detected_lang
        new_slots = nlu.get("slots", {})
        self.update_slots(new_slots)

        response = self._handle(intent, user_message, new_slots)
        self.history.append({"role": "assistant", "message": response})
        return response

    def _handle(self, intent: str, user_msg: str, new_slots: dict = {}) -> str:

        # ── Transitions globales ─────────────────────────────────────
        if intent == "au_revoir":
            self.state = "GOODBYE"
            return self._gen("au_revoir", user_msg=user_msg)

        if intent == "out_of_scope":
            return self.OUT_OF_SCOPE.get(self.lang, self.OUT_OF_SCOPE["fr"])

        if intent == "demande_clarification":
            resp = self.CLARIFICATION_ANSWERS.get(self.lang, self.CLARIFICATION_ANSWERS["fr"])
            if self.current_flight:
                resp += "\n\n" + format_flight_details(self.current_flight, self.lang)
            return resp


        if intent == "correction_slot":
          # 从 new_slots 里提取修正的城市
            if new_slots.get("to_city"):
                self.slots["to_city"] = new_slots["to_city"]
            if new_slots.get("from_city"):
                self.slots["from_city"] = new_slots["from_city"]
            self.excluded_ids = set()
            self.current_flight = None
            return self._try_search(user_msg)

        # ── États ────────────────────────────────────────────────────
        if self.state == "INIT":
            self.state = "GREETING"
            if intent == "salutation":
                return self._gen("salutation", user_msg=user_msg)
            elif intent in ("flight", "airfare", "flight_time"):
                return self._collect_and_search(user_msg)
            return self._gen("salutation", user_msg=user_msg)

        elif self.state == "GREETING":
            if intent in ("flight", "airfare", "flight_time", "airline", "aircraft"):
                return self._collect_and_search(user_msg)
            return self._gen("salutation", user_msg=user_msg)

        elif self.state == "COLLECTING":
            return self._collect_and_search(user_msg)

        elif self.state == "RECOMMENDATION":
            if intent == "accepter":
                return self._confirm_booking()
            elif intent == "rejeter":
                return self._next_flight(user_msg)
            elif intent == "flight_time":
                return self._next_flight(user_msg)
            elif intent == "demande_details" or re.search(
                r"(détail|detail|plus d'info|tell me more|详细|介绍)", user_msg.lower()
            ):
                self.state = "DETAILS"
                if self.current_flight:
                    details = format_flight_details(self.current_flight, self.lang)
                    nlg = self._gen("demande_details", self.current_flight, user_msg)
                    return f"{nlg}\n\n{details}"
                return self._gen("incomprehension", user_msg=user_msg)
            elif intent in ("flight", "airfare"):
                return self._collect_and_search(user_msg)
            elif any(k in self.slots for k in ("from_city", "to_city")):
                return self._try_search(user_msg)
            return self._gen("demande_details", self.current_flight, user_msg)

        elif self.state == "DETAILS":
            if intent == "accepter":
                return self._confirm_booking()
            elif intent == "rejeter":
                return self._next_flight(user_msg)
            elif intent in ("flight", "airfare", "flight_time"):
                return self._collect_and_search(user_msg)
            # ── 新增：用户提供新城市 → 重置重新搜索 ──────────────────
            elif new_slots.get("from_city") or new_slots.get("to_city"):
                self.slots = {k: v for k, v in new_slots.items()}
                self.excluded_ids = set()
                self.current_flight = None
                return self._collect_and_search(user_msg)
            # ─────────────────────────────────────────────────────────
            self.state = "FOLLOW_UP"
            return self._gen("accepter", self.current_flight, user_msg)

        elif self.state == "BOOKING_CONFIRMED":
            self.state = "FOLLOW_UP"
            return self._gen("au_revoir", user_msg=user_msg)

        elif self.state == "FOLLOW_UP":
            if intent in ("flight", "airfare", "flight_time"):
                self.slots = {}
                self.excluded_ids = set()
                self.current_flight = None
                return self._collect_and_search(user_msg)
            elif intent == "salutation":
                return self._gen("salutation", user_msg=user_msg)
            return self._gen("incomprehension", user_msg=user_msg)

        elif self.state == "GOODBYE":
            self.reset()
            return self._gen("salutation", user_msg=user_msg)

        return self._gen("incomprehension", user_msg=user_msg)



    def _collect_and_search(self, user_msg: str) -> str:
        """Vérifie les slots obligatoires, demande si manquants, sinon cherche."""
        if not self.slots.get("from_city"):
            self.state = "COLLECTING"
            return self.MISSING_FROM.get(self.lang, self.MISSING_FROM["fr"])
        if not self.slots.get("to_city"):
            self.state = "COLLECTING"
            return self.MISSING_TO.get(self.lang, self.MISSING_TO["fr"])
        return self._try_search(user_msg)

    def _try_search(self, user_msg: str) -> str:
        """Lance la recherche et présente le premier résultat."""
        results = self._do_search()
        if not results:
            # ── BUG FIX #2 ──────────────────────────────────────────
            # Si aucun résultat ET qu'on avait un vol courant, on le conserve
            # et on informe l'utilisateur plutôt que de casser l'état.
            if self.current_flight:
                self.state = "RECOMMENDATION"
                msg = self.NO_RESULTS.get(self.lang, self.NO_RESULTS["fr"])
                flight_card = format_flight_short(self.current_flight, self.lang)
                options = self._options_menu()
                return f"{msg}\n\n{flight_card}{options}"
            # ── END FIX #2 ──────────────────────────────────────────
            self.state = "COLLECTING"
            return self.NO_RESULTS.get(self.lang, self.NO_RESULTS["fr"])

        self.current_flight = results[0]
        self.excluded_ids.add(self.current_flight["flight_id"])
        self.state = "RECOMMENDATION"

        flight_card = format_flight_short(self.current_flight, self.lang)
        nlg = self._gen("flight", self.current_flight, user_msg)
        if self.current_flight["flight_id"] not in nlg:
            nlg = flight_card + "\n\n" + nlg.split("\n\n")[0] if "\n\n" in nlg else flight_card
        return nlg + self._options_menu()

    def _options_menu(self) -> str:
        """Retourne le menu BF=7 dans la bonne langue."""
        options = {
            "fr": (
                "\n\nQue souhaitez-vous faire ?\n"
                "1️⃣ Réserver ce vol\n"
                "2️⃣ Voir un vol moins cher\n"
                "3️⃣ Voir un vol plus tôt\n"
                "4️⃣ Voir un vol plus tard\n"
                "5️⃣ Changer la compagnie\n"
                "6️⃣ Voir les détails complets\n"
                "7️⃣ Chercher une autre destination"
            ),
            "en": (
                "\n\nWhat would you like to do?\n"
                "1️⃣ Book this flight\n"
                "2️⃣ Find a cheaper flight\n"
                "3️⃣ Find an earlier flight\n"
                "4️⃣ Find a later flight\n"
                "5️⃣ Change the airline\n"
                "6️⃣ See full details\n"
                "7️⃣ Search a different destination"
            ),
            "zh": (
                "\n\n您想做什么？\n"
                "1️⃣ 预订这个航班\n"
                "2️⃣ 寻找更便宜的航班\n"
                "3️⃣ 寻找更早的航班\n"
                "4️⃣ 寻找更晚的航班\n"
                "5️⃣ 更换航空公司\n"
                "6️⃣ 查看完整详情\n"
                "7️⃣ 搜索其他目的地"
            ),
        }
        return options.get(self.lang, options["fr"])

    def _next_flight(self, user_msg: str) -> str:
        """Propose le vol suivant. Si plus de résultats, informe sans casser l'état."""
        remaining = [f for f in self.last_results if f["flight_id"] not in self.excluded_ids]
        if remaining:
            self.current_flight = remaining[0]
            self.excluded_ids.add(self.current_flight["flight_id"])
            self.state = "RECOMMENDATION"
            nlg = self._gen("flight", self.current_flight, user_msg)
            if self.current_flight["flight_id"] not in nlg:
                nlg = format_flight_short(self.current_flight, self.lang) + "\n\n" + nlg
            return nlg + self._options_menu()

        # ── BUG FIX #3 ──────────────────────────────────────────────
        # Plus d'alternatives : on réessaie SANS le flight_mod pour ne pas
        # bloquer sur "plus tôt" quand il n'y a qu'un seul vol sur la route.
        # On conserve current_flight si la recherche élargie échoue aussi.
        saved_flight = self.current_flight
        flight_mod = self.slots.pop("flight_mod", None)  # retire temporairement
        cost_rel   = self.slots.pop("cost_relative", None)

        results = search_flights(
            self.slots, self.flights_db, self.lexique,
            exclude_ids=self.excluded_ids, top_n=7
        )

        if results:
            self.last_results = results
            self.current_flight = results[0]
            self.excluded_ids.add(self.current_flight["flight_id"])
            self.state = "RECOMMENDATION"
            # Message contextuel selon le modificateur demandé
            prefix = ""
            if flight_mod == "premier_disponible":
                prefix = self.NO_EARLIER.get(self.lang, self.NO_EARLIER["fr"]) + "\n\n"
            elif flight_mod == "dernier_disponible":
                prefix = self.NO_LATER.get(self.lang, self.NO_LATER["fr"]) + "\n\n"
            nlg = self._gen("flight", self.current_flight, user_msg)
            if self.current_flight["flight_id"] not in nlg:
                nlg = format_flight_short(self.current_flight, self.lang) + "\n\n" + nlg
            return prefix + nlg + self._options_menu()

        # Vraiment aucun résultat : on remet les slots et on garde le vol courant
        if flight_mod:
            self.slots["flight_mod"] = flight_mod
        if cost_rel:
            self.slots["cost_relative"] = cost_rel
        self.current_flight = saved_flight

        if self.current_flight:
            self.state = "RECOMMENDATION"
            msg = self.NO_RESULTS.get(self.lang, self.NO_RESULTS["fr"])
            return msg + "\n\n" + format_flight_short(self.current_flight, self.lang) + self._options_menu()

        self.state = "COLLECTING"
        return self.NO_RESULTS.get(self.lang, self.NO_RESULTS["fr"])
        # ── END FIX #3 ──────────────────────────────────────────────

    def _confirm_booking(self) -> str:
        """Confirme la réservation."""
        if not self.current_flight:
            return self._gen("incomprehension")
        self.state = "BOOKING_CONFIRMED"
        f = self.current_flight
        msg = self.BOOKING_CONFIRMED_MSG.get(self.lang, self.BOOKING_CONFIRMED_MSG["fr"])
        return msg.format(
            fid=f["flight_id"],
            airline=f["airline"],
            from_city=f["from_city"],
            to_city=f["to_city"],
            depart=f["depart_time"],
        )

print("✓ DialogManager chargé (v2 — bugs BF=7 corrigés)")
print(f"  États : {DialogManager.STATES}")

✓ DialogManager chargé (v2 — bugs BF=7 corrigés)
  États : ['INIT', 'GREETING', 'COLLECTING', 'RECOMMENDATION', 'DETAILS', 'BOOKING', 'BOOKING_CONFIRMED', 'FOLLOW_UP', 'GOODBYE']


## 9. Interface Gradio et enregistrement des conversations

L'interface web est construite avec Gradio et inclut :
- Chat multilingue (FR / EN / ZH)  
- Bouton de réinitialisation  
- Statistiques en temps réel  
- Enregistrement automatique des conversations (bonus)


In [130]:
CONVERSATION_LOGS = []
dm = DialogManager(flights_db, lexique)


def save_conversation_log(log: dict):
    """Sauvegarde la conversation dans un fichier JSON."""
    CONVERSATION_LOGS.append(log)
    log_dir = "/content/drive/MyDrive/conversation/historique"
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath = os.path.join(log_dir, f"conv_{timestamp}.json")
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)
    print(f"  [Log] Conversation sauvegardée : {filepath}")


def chat_respond(message: str, history: list) -> Tuple[str, list]:
    if not message.strip():
        return "", history
    response = dm.process(message.strip())
    history.append((message, response))
    if dm.state in ("GOODBYE", "BOOKING_CONFIRMED"):
        save_conversation_log(dm.get_conversation_log())
    return "", history


def reset_chat():
    if dm.turn_count > 0:
        save_conversation_log(dm.get_conversation_log())
    dm.reset()
    return [], ""


def get_stats():
    routes = {}
    for f in flights_db:
        r = f"{f['from_city']} → {f['to_city']}"
        routes[r] = routes.get(r, 0) + 1
    top_routes = sorted(routes.items(), key=lambda x: x[1], reverse=True)[:5]
    airlines_count = {}
    for f in flights_db:
        airlines_count[f["airline"]] = airlines_count.get(f["airline"], 0) + 1

    s  = "📊 **Statistiques VoyageAssistant**\n\n"
    s += f"- Vols en base : **{len(flights_db)}**\n"
    s += f"- Villes desservies : **{len(cities_db)}**\n"
    s += f"- Compagnies : **{len(airlines_db)}**\n"
    s += f"- Conversations enregistrées : **{len(CONVERSATION_LOGS)}**\n"
    s += f"- Tours actuels : **{dm.turn_count}**\n"
    s += f"- État actuel : **{dm.state}**\n\n"
    s += "**Top 5 routes :**\n"
    for route, cnt in top_routes:
        s += f"- {route} : {cnt} vols\n"
    return s


with gr.Blocks(title="VoyageAssistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
    "# ✈️ VoyageAssistant — Réservation de Vols Multilingue\n"
    "Recherchez et réservez vos vols en **français**, **anglais** ou **中文**.\n\n"
    "Search and book flights in **French**, **English** or **Chinese**.\n\n"
    "💾 *Pour sauvegarder votre conversation, dites* **« Au revoir »** *ou confirmez une réservation.*"
)
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Conversation", height=520, show_copy_button=True)
            with gr.Row():
                msg_input = gr.Textbox(
                    label="Votre message",
                    placeholder="Ex: Je veux un vol de Paris à Tokyo / I need a flight to NYC / 我想从北京飞上海",
                    scale=4, show_label=False,
                )
                send_btn = gr.Button("Envoyer ✈️", variant="primary", scale=1)
            with gr.Row():
                reset_btn = gr.Button("🔄 Nouvelle conversation", variant="secondary")

        with gr.Column(scale=1):
            stats_display = gr.Markdown(get_stats())
            refresh_btn = gr.Button("🔄 Rafraîchir")
            gr.Markdown(
                "### 💡 Exemples\n"
                "**Français :**\n"
                "- Bonjour !\n"
                "- Vol de Paris à Tokyo\n"
                "- Un vol pas cher direct\n"
                "- Aller-retour New York le 15 mars\n\n"
                "**English:**\n"
                "- Hi!\n"
                "- Flight from London to NYC\n"
                "- Cheapest nonstop flight\n\n"
                "**中文：**\n"
                "- 你好！\n"
                "- 北京到上海的航班\n"
                "- 最便宜的直飞\n"
            )

    send_btn.click(chat_respond, [msg_input, chatbot], [msg_input, chatbot])
    msg_input.submit(chat_respond, [msg_input, chatbot], [msg_input, chatbot])
    reset_btn.click(reset_chat, outputs=[chatbot, msg_input])
    refresh_btn.click(get_stats, outputs=[stats_display])

print("✓ Interface Gradio configurée")


/tmp/ipykernel_768/1764598367.py:57: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="VoyageAssistant", theme=gr.themes.Soft()) as demo:


✓ Interface Gradio configurée


/tmp/ipykernel_768/1764598367.py:66: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation", height=520, show_copy_button=True)
/tmp/ipykernel_768/1764598367.py:66: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(label="Conversation", height=520, show_copy_button=True)
/tmp/ipykernel_768/1764598367.py:66: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation", height=520, show_copy_button=True)

## 10. Lancement de l'application

Exécutez la cellule ci-dessous pour démarrer VoyageAssistant.  
Sur Google Colab, un lien public `share=True` sera généré automatiquement.


In [131]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a47c885b85ab2e5440.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Log] Conversation sauvegardée : /content/drive/MyDrive/conversation/historique/conv_20260306_093730.json


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a47c885b85ab2e5440.gradio.live


## 11. Analyse des conversations enregistrées (bonus)

Cette section permet d'analyser les conversations pour évaluer les performances du système.


In [132]:
def analyze_conversations():
    log_dir = "conversation_logs"
    if not os.path.exists(log_dir):
        print("Aucune conversation enregistrée.")
        return

    files = [f for f in os.listdir(log_dir) if f.endswith(".json")]
    print(f"\n📊 Analyse de {len(files)} conversation(s)\n")

    total_turns = 0
    booked = 0
    langs = {}
    intents_count = {}

    for fname in files:
        with open(os.path.join(log_dir, fname), "r", encoding="utf-8") as f:
            conv = json.load(f)
        total_turns += conv.get("turns", 0)
        if conv.get("flight_booked"):
            booked += 1
        lang = conv.get("lang", "fr")
        langs[lang] = langs.get(lang, 0) + 1
        for turn in conv.get("history", []):
            if turn["role"] == "user":
                nlu = rule_based_nlu(turn["message"], lexique)
                if nlu:
                    intent = nlu.get("intent", "unknown")
                    intents_count[intent] = intents_count.get(intent, 0) + 1

    print(f"  Conversations totales   : {len(files)}")
    print(f"  Réservations effectuées : {booked} ({100*booked//max(len(files),1)}%)")
    print(f"  Tours moyens/conv       : {total_turns // max(len(files), 1)}")
    print(f"  Langues utilisées       : {langs}")
    print(f"  Intents détectés        : {intents_count}")

analyze_conversations()


Aucune conversation enregistrée.


In [133]:
# Export des logs
import shutil
try:
    from google.colab import files
    shutil.make_archive("conversation_logs", "zip", ".", "conversation_logs")
    files.download("conversation_logs.zip")
except ImportError:
    print("Export disponible uniquement sur Google Colab.")


FileNotFoundError: [Errno 2] No such file or directory: 'conversation_logs'